In [2]:
# Código: Dairo Alberto Herrera Escobar
# Paso 1: cargar los datos

import pandas as pd

datos = pd.read_csv("Saber11_Matematicas.csv")

print("Filas y columnas:", datos.shape)
datos.head()

Filas y columnas: (480381, 18)


,cole_area_ubicacion,cole_depto_ubicacion,cole_jornada,cole_naturaleza,estu_dedicacionlecturadiaria,estu_horassemanatrabaja,fami_cuartoshogar,fami_educacionmadre,fami_educacionpadre,fami_estratovivienda,fami_numlibros,fami_personashogar,fami_tienecomputador,fami_tieneinternet,estu_horastrabnoremu,estu_desplazacolegio,edad,punt_matematicas
0,URBANO,BOGOTÁ,TARDE,OFICIAL,No leo por entretenimiento,0,Tres,Primaria incompleta,No sabe,Estrato 2,0 A 10 LIBROS,5 a 6,No,No,Más de 30 horas,En bus o transporte público,17,49
1,URBANO,BOGOTÁ,UNICA,OFICIAL,No leo por entretenimiento,0,Tres,Técnica o tecnológica completa,Secundaria (Bachillerato) completa,Estrato 2,11 A 25 LIBROS,3 a 4,Si,Si,Menos de 10 horas,Caminando o en bicicleta,16,64
2,URBANO,ANTIOQUIA,MAÑANA,OFICIAL,No leo por entretenimiento,0,Dos,Primaria incompleta,Primaria incompleta,Estrato 1,0 A 10 LIBROS,3 a 4,Si,Si,0,Caminando o en bicicleta,18,53
3,URBANO,ANTIOQUIA,TARDE,OFICIAL,30 minutos o menos,0,Dos,Secundaria (Bachillerato) completa,Secundaria (Bachillerato) completa,Estrato 2,26 A 100 LIBROS,3 a 4,Si,Si,Menos de 10 horas,Caminando o en bicicleta,17,64
4,URBANO,ATLANTICO,UNICA,OFICIAL,Entre 1 y 2 horas,0,Tres,Secundaria (Bachillerato) completa,Secundaria (Bachillerato) completa,Estrato 1,11 A 25 LIBROS,5 a 6,Si,Si,0,Caminando o en bicicleta,16,48


In [3]:
# Código: Dairo Alberto Herrera Escobar
# Paso 2: dummies (convertir texto en columnas 0/1)

# Columnas de texto (todas menos edad y punt_matematicas)
columnas_texto = datos.select_dtypes(include="object").columns.tolist()
print("Columnas de texto:", len(columnas_texto))

datos_dummies = pd.get_dummies(datos, columns=columnas_texto, dtype=int)

print("Columnas antes:", datos.shape[1])
print("Columnas después:", datos_dummies.shape[1])
datos_dummies.head()

Columnas de texto: 16
Columnas antes: 18
Columnas después: 114


,edad,punt_matematicas,cole_area_ubicacion_RURAL,cole_area_ubicacion_URBANO,cole_depto_ubicacion_AMAZONAS,cole_depto_ubicacion_ANTIOQUIA,cole_depto_ubicacion_ARAUCA,cole_depto_ubicacion_ATLANTICO,cole_depto_ubicacion_BOGOTÁ,cole_depto_ubicacion_BOLIVAR,...,fami_tieneinternet_Si,estu_horastrabnoremu_0,estu_horastrabnoremu_Entre 11 y 20 horas,estu_horastrabnoremu_Entre 21 y 30 horas,estu_horastrabnoremu_Menos de 10 horas,estu_horastrabnoremu_Más de 30 horas,estu_desplazacolegio_Caminando o en bicicleta,estu_desplazacolegio_En bus o transporte público,estu_desplazacolegio_En transporte escolar o particular,"estu_desplazacolegio_Otro (burro o caballo, canoa, lancha, etc.)"
0,17,49,0,1,0,0,0,0,1,0,...,0,0,0,0,0,1,0,1,0,0
1,16,64,0,1,0,0,0,0,1,0,...,1,0,0,0,1,0,1,0,0,0
2,18,53,0,1,0,1,0,0,0,0,...,1,1,0,0,0,0,1,0,0,0
3,17,64,0,1,0,1,0,0,0,0,...,1,0,0,0,1,0,1,0,0,0
4,16,48,0,1,0,0,0,1,0,0,...,1,1,0,0,0,0,1,0,0,0


In [4]:
# Código: Dairo Alberto Herrera Escobar
# Paso 3: separar entrenamiento (80 %) y prueba (20 %)

from sklearn.model_selection import train_test_split

X = datos_dummies.drop(columns="punt_matematicas")   # pistas
y = datos_dummies["punt_matematicas"]                # objetivo

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)

Entrenamiento: (384304, 113)
Prueba: (96077, 113)


In [5]:
# Código: Dairo Alberto Herrera Escobar
# Paso 4: normalizar la edad (escala 0 a 1)

from sklearn.preprocessing import MinMaxScaler

escalador = MinMaxScaler()

X_train_norm = X_train.copy()
X_test_norm = X_test.copy()

# Aprende la escala SOLO con entrenamiento y la aplica
X_train_norm[["edad"]] = escalador.fit_transform(X_train[["edad"]])
# Aplica la MISMA escala a prueba
X_test_norm[["edad"]] = escalador.transform(X_test[["edad"]])

print("Edad original (entrenamiento): mín", X_train["edad"].min(), "- máx", X_train["edad"].max())
print("Edad normalizada (entrenamiento): mín", X_train_norm["edad"].min(), "- máx", X_train_norm["edad"].max())

Edad original (entrenamiento): mín 12 - máx 87
Edad normalizada (entrenamiento): mín 0.0 - máx 1.0000000000000002


In [6]:
# Código: Dairo Alberto Herrera Escobar
# Paso 5: línea base (decir a todos el puntaje promedio)

from sklearn.dummy import DummyRegressor
from sklearn.model_selection import KFold, cross_validate

# Validación cruzada: 5 montoncitos (se usará igual en todos los modelos)
particion = KFold(n_splits=5, shuffle=True, random_state=42)

linea_base = DummyRegressor(strategy="mean")

resultado = cross_validate(
    linea_base, X_train, y_train, cv=particion,
    scoring=["neg_mean_absolute_error", "neg_root_mean_squared_error"]
)

mae_base = -resultado["test_neg_mean_absolute_error"].mean()
rmse_base = -resultado["test_neg_root_mean_squared_error"].mean()

print("Promedio que se le dice a todos:", round(y_train.mean(), 2))
print("Línea base - MAE:", round(mae_base, 2))
print("Línea base - RMSE:", round(rmse_base, 2))

Promedio que se le dice a todos: 52.48
Línea base - MAE: 10.69
Línea base - RMSE: 13.18


In [7]:
# Código: Dairo Alberto Herrera Escobar
# Paso 6: Modelo 1 - Regresión lineal

import time
from sklearn.linear_model import LinearRegression

resultados = []   # aquí iremos guardando todos los modelos

modelo = LinearRegression()

inicio = time.time()
res = cross_validate(
    modelo, X_train_norm, y_train, cv=particion,
    scoring=["neg_mean_absolute_error", "neg_root_mean_squared_error"]
)
tiempo = time.time() - inicio

mae = -res["test_neg_mean_absolute_error"].mean()
rmse = -res["test_neg_root_mean_squared_error"].mean()
resultados.append(["Regresión lineal", "Por defecto", mae, rmse, tiempo])

print("Regresión lineal - MAE:", round(mae, 2))
print("Regresión lineal - RMSE:", round(rmse, 2))
print("Tiempo (segundos):", round(tiempo, 1))
print("¿Le gana a la línea base?", mae < mae_base)

Regresión lineal - MAE: 9.18
Regresión lineal - RMSE: 11.48
Tiempo (segundos): 25.3
¿Le gana a la línea base? True


In [8]:
# Código: Dairo Alberto Herrera Escobar
# Paso 7: Modelo 2 - Árbol de regresión (por defecto)

from sklearn.tree import DecisionTreeRegressor

modelo = DecisionTreeRegressor(random_state=42)

inicio = time.time()
res = cross_validate(
    modelo, X_train, y_train, cv=particion,
    scoring=["neg_mean_absolute_error", "neg_root_mean_squared_error"],
    return_train_score=True
)
tiempo = time.time() - inicio

mae = -res["test_neg_mean_absolute_error"].mean()
rmse = -res["test_neg_root_mean_squared_error"].mean()
mae_entreno = -res["train_neg_mean_absolute_error"].mean()
resultados.append(["Árbol de regresión", "Por defecto", mae, rmse, tiempo])

print("MAE en entrenamiento (lo que ya vio):", round(mae_entreno, 2))
print("MAE en validación (lo que NO vio):", round(mae, 2))
print("RMSE en validación:", round(rmse, 2))
print("Tiempo (segundos):", round(tiempo, 1))
print("¿Le gana a la línea base?", mae < mae_base)

MAE en entrenamiento (lo que ya vio): 0.18
MAE en validación (lo que NO vio): 12.82
RMSE en validación: 16.14
Tiempo (segundos): 67.6
¿Le gana a la línea base? False


In [10]:
# Código: Dairo Alberto Herrera Escobar
# Paso 8: Árbol de regresión - probar la perilla max_depth

from sklearn.tree import DecisionTreeRegressor

valores = [10,15,20]   # <-- ESCRIBE AQUÍ TUS VALORES, separados por coma. Ej: [a, b, c, d]

for v in valores:
    modelo = DecisionTreeRegressor(max_depth=v, random_state=42)
    inicio = time.time()
    res = cross_validate(modelo, X_train, y_train, cv=particion,
                         scoring=["neg_mean_absolute_error", "neg_root_mean_squared_error"])
    tiempo = time.time() - inicio
    mae = -res["test_neg_mean_absolute_error"].mean()
    rmse = -res["test_neg_root_mean_squared_error"].mean()
    resultados.append(["Árbol de regresión", f"max_depth={v}", mae, rmse, tiempo])
    print(f"max_depth={v}  ->  MAE: {mae:.2f}   RMSE: {rmse:.2f}   Tiempo: {tiempo:.1f} s")

max_depth=10  ->  MAE: 9.39   RMSE: 11.73   Tiempo: 23.2 s
max_depth=15  ->  MAE: 9.47   RMSE: 11.90   Tiempo: 32.6 s
max_depth=20  ->  MAE: 9.88   RMSE: 12.51   Tiempo: 40.5 s


In [11]:
# Código: Dairo Alberto Herrera Escobar
# Paso 9: Modelo 3 - Gradient Boosting (por defecto)

from sklearn.ensemble import GradientBoostingRegressor

modelo = GradientBoostingRegressor(random_state=42)

inicio = time.time()
res = cross_validate(
    modelo, X_train, y_train, cv=particion,
    scoring=["neg_mean_absolute_error", "neg_root_mean_squared_error"],
    n_jobs=-1
)
tiempo = time.time() - inicio

mae = -res["test_neg_mean_absolute_error"].mean()
rmse = -res["test_neg_root_mean_squared_error"].mean()
resultados.append(["Gradient Boosting", "Por defecto", mae, rmse, tiempo])

print("Gradient Boosting - MAE:", round(mae, 2))
print("Gradient Boosting - RMSE:", round(rmse, 2))
print("Tiempo (segundos):", round(tiempo, 1))
print("¿Le gana a la línea base?", mae < mae_base)

Gradient Boosting - MAE: 9.13
Gradient Boosting - RMSE: 11.39
Tiempo (segundos): 573.4
¿Le gana a la línea base? True


In [12]:
# Código: Dairo Alberto Herrera Escobar
# Paso 10: Modelo 4 - Red neuronal (parada temprana)

from sklearn.neural_network import MLPRegressor

modelo = MLPRegressor(early_stopping=True, random_state=42)

inicio = time.time()
res = cross_validate(
    modelo, X_train_norm, y_train, cv=particion,
    scoring=["neg_mean_absolute_error", "neg_root_mean_squared_error"],
    n_jobs=-1
)
tiempo = time.time() - inicio

mae = -res["test_neg_mean_absolute_error"].mean()
rmse = -res["test_neg_root_mean_squared_error"].mean()
resultados.append(["Red neuronal", "early_stopping=True", mae, rmse, tiempo])

print("Red neuronal - MAE:", round(mae, 2))
print("Red neuronal - RMSE:", round(rmse, 2))
print("Tiempo (segundos):", round(tiempo, 1))
print("¿Le gana a la línea base?", mae < mae_base)

Red neuronal - MAE: 8.91
Red neuronal - RMSE: 11.18
Tiempo (segundos): 353.6
¿Le gana a la línea base? True


In [13]:
# Código: Dairo Alberto Herrera Escobar
# Paso 11: Tabla comparativa de todos los modelos (validación cruzada)

tabla = pd.DataFrame(resultados, columns=["Método", "Configuración", "MAE", "RMSE", "Tiempo (s)"])

# Agregar la línea base para comparar
tabla.loc[len(tabla)] = ["Línea base", "Promedio", mae_base, rmse_base, 0]

# Cuánto mejora cada modelo frente a la línea base (en %)
tabla["Mejora vs línea base (%)"] = (mae_base - tabla["MAE"]) / mae_base * 100

tabla = tabla.drop_duplicates().sort_values("MAE").reset_index(drop=True)
tabla.round(2)

,Método,Configuración,MAE,RMSE,Tiempo (s),Mejora vs línea base (%)
0,Red neuronal,early_stopping=True,8.91,11.18,353.64,16.64
1,Gradient Boosting,Por defecto,9.13,11.39,573.44,14.59
2,Regresión lineal,Por defecto,9.18,11.48,25.27,14.16
3,Árbol de regresión,max_depth=10,9.39,11.73,23.23,12.20
4,Árbol de regresión,max_depth=15,9.47,11.90,32.64,11.41
5,Árbol de regresión,max_depth=7,9.50,11.85,16.84,11.12
6,Árbol de regresión,max_depth=5,9.66,12.03,12.21,9.66
7,Árbol de regresión,max_depth=20,9.88,12.51,40.50,7.61
8,Árbol de regresión,max_depth=3,9.94,12.33,7.38,7.07
9,Línea base,Promedio,10.69,13.18,0.00,0.00


In [14]:
# Código: Dairo Alberto Herrera Escobar
# Paso 12: Examen final del mejor modelo (red neuronal) con el 20 % de prueba

from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import joblib

# 1. Entrenar con todo el entrenamiento
mejor_modelo = MLPRegressor(early_stopping=True, random_state=42)
inicio = time.time()
mejor_modelo.fit(X_train_norm, y_train)
print("Tiempo de entrenamiento final (s):", round(time.time() - inicio, 1))

# 2. Examen final con los datos de prueba
pred = mejor_modelo.predict(X_test_norm)
mae_final = mean_absolute_error(y_test, pred)
rmse_final = np.sqrt(mean_squared_error(y_test, pred))

# Línea base en el mismo examen (decir a todos el promedio del entrenamiento)
pred_base = np.full(len(y_test), y_train.mean())
mae_base_test = mean_absolute_error(y_test, pred_base)
rmse_base_test = np.sqrt(mean_squared_error(y_test, pred_base))

print("EXAMEN FINAL - Red neuronal:  MAE", round(mae_final, 2), "| RMSE", round(rmse_final, 2))
print("EXAMEN FINAL - Línea base:    MAE", round(mae_base_test, 2), "| RMSE", round(rmse_base_test, 2))

# 3. Guardar el modelo, el escalador y las columnas (para Streamlit)
joblib.dump(mejor_modelo, "modelo_red_neuronal.pkl")
joblib.dump(escalador, "escalador_edad.pkl")
joblib.dump(list(X_train.columns), "columnas_modelo.pkl")
print("Archivos guardados.")

Tiempo de entrenamiento final (s): 135.6
EXAMEN FINAL - Red neuronal:  MAE 8.91 | RMSE 11.17
EXAMEN FINAL - Línea base:    MAE 10.71 | RMSE 13.2
Archivos guardados.
